# Testing Queries for Seiscomp6

In previous versions of the revision routine, there are a fixed set of queries that are used to test the connection to the database and to extract the info needed to run the revision. Here, we test the same queries for Seiscomp6, and make sure they work as expected. We will also test the connection to the database, and make sure that we can ping the server. In addition, by using DBeaver I try to extend the queries to extract more info that may be useful for the revision, mainly for those events with 6 to 8 phases.

Let's creating a single function that queries some info from the seiscomp database.

In [12]:
import datetime
import os
import warnings
import pymysql
import pandas as pd
import datetime as dt
from tqdm import tqdm
from dotenv import load_dotenv

env_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path=env_path)

def connect_to_db(
        query: str,
        start_time: dt.datetime = None,
        end_time: dt.datetime = None,
        **kwargs):

    if start_time and end_time:
        start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")
        full_query = f"{query} '{start_time_str}' and '{end_time_str}'"
    else:
        full_query = query

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        db_connection = pymysql.connect(
            host=os.getenv('SERVER_HOST'),
            user=os.getenv('SERVER_USERNAME'),
            password=os.getenv('SERVER_PASSWORD'),
            db=os.getenv('SERVER_DATABASE')
        )

        try:
            with tqdm(total=1, desc='Querying database...', unit='query', leave=False,bar_format="{desc}") as pbar:
                df = pd.read_sql_query(full_query, db_connection, **kwargs)
                pbar.update(1)
        finally:
            db_connection.close()

    return df

# Unboxing current queries

Let's start by unboxing the current queries used in the revision routine, and test them one by one.

## Normal queries

The SQL query used is:

```sql
Select Origin.time_value, POEv.publicID, Origin.depth_value, Magnitude.magnitude_value, Origin.quality_standardError, Origin.depth_uncertainty, Origin.latitude_uncertainty, Origin.longitude_uncertainty, Origin.quality_associatedPhaseCount, Origin.creationInfo_author, Event.type, Origin.creationInfo_agencyID, EventDescription.text, Origin.latitude_value, Origin.longitude_value, Magnitude.type, Origin.methodID, Origin.earthModelID from Event AS EvMF left join PublicObject AS POEv ON EvMF._oid = POEv._oid left join PublicObject as POOri ON EvMF.preferredOriginID=POOri.publicID left join Origin ON POOri._oid=Origin._oid left join PublicObject as POMag on EvMF.preferredMagnitudeID=POMag.publicID left join Magnitude ON Magnitude._oid = POMag._oid left join Event ON Event._oid= POEv._oid left join EventDescription ON EvMF._oid = EventDescription._parent_oid where Origin.time_value between
```

where:

- `EvMF` is the Event table, which contains the main information about the event, such as the origin time, depth, magnitude, etc.
- `POEv` is the PublicObject table, which contains the public ID of the event, which is used to link the Event table with the Origin and Magnitude tables.
- `POOri` is the PublicObject table, which contains the public ID of the origin, which is used to link the Origin table with the Event table.
- `Origin` is the Origin table, which contains the information about the origin of the event, such as the time, depth, latitude, longitude, etc.
- `POMag` is the PublicObject table, which contains the public ID of the magnitude, which is used to link the Magnitude table with the Event table.
- `Magnitude` is the Magnitude table, which contains the information about the magnitude of the event, such as the magnitude value, type, etc.
- `EventDescription` is the EventDescription table, which contains the description of the event, such as the text description, etc.

In this SQL query, we are selecting the following fields:
- `Origin.time_value`: the origin time of the event
- `POEv.publicID`: the public ID of the event
- `Origin.depth_value`: the depth of the event
- `Magnitude.magnitude_value`: the magnitude of the event
- `Origin.quality_standardError`: the standard error of the origin
- `Origin.depth_uncertainty`: the uncertainty of the depth
- `Origin.latitude_uncertainty`: the uncertainty of the latitude
- `Origin.longitude_uncertainty`: the uncertainty of the longitude
- `Origin.quality_associatedPhaseCount`: the number of associated phases
- `Origin.creationInfo_author`: the author of the origin
- `Event.type`: the type of the event
- `Origin.creationInfo_agencyID`: the agency ID of the origin
- `EventDescription.text`: the text description of the event
- `Origin.latitude_value`: the latitude of the event
- `Origin.longitude_value`: the longitude of the event
- `Magnitude.type`: the type of the magnitude
- `Origin.methodID`: the method ID of the origin
- `Origin.earthModelID`: the earth model ID of the origin

and the condition is that the origin time of the event is between a certain time range, which is specified in the last part of the query.

Let's test this query directly in this notebook, by using the function from the last cell. We will see as a simple example the fist ten columns of the ORIGIN table, which contains the information about the origin of the events. We will also test the query for a specific time range, to see if we can retrieve the events that occurred in that time range.

In [27]:
query_example = "SELECT * FROM Origin WHERE Origin.time_value BETWEEN"
origin_df = connect_to_db(query_example, start_time=dt.datetime(2026, 6, 4, 14, 0, 0), end_time=dt.datetime(2026, 6, 5, 1, 0, 0))#dt.datetime.now(dt.UTC))
origin_df

2026-06-04 14:00:00 2026-06-05 01:00:00


,_oid,_parent_oid,_last_modified,time_value,time_value_ms,time_uncertainty,time_lowerUncertainty,time_upperUncertainty,time_confidenceLevel,time_pdf_variable_content,...,creationInfo_agencyID,creationInfo_agencyURI,creationInfo_author,creationInfo_authorURI,creationInfo_creationTime,creationInfo_creationTime_ms,creationInfo_modificationTime,creationInfo_modificationTime_ms,creationInfo_version,creationInfo_used
0,766634671,1,2026-06-04 23:15:01,2026-06-04 22:18:08,982210,1.381229,None,None,None,None,...,SGC,,scanloc,,2026-06-04 22:20:05,780592,NaT,NaN,,1
1,766634797,1,2026-06-04 23:15:01,2026-06-04 22:18:08,982210,1.381229,None,None,None,None,...,SGC,,scanloc,,2026-06-04 22:20:36,627900,NaT,NaN,,1
2,766634734,1,2026-06-04 23:15:01,2026-06-04 22:18:09,14357,1.380996,None,None,None,None,...,SGC,,scanloc,,2026-06-04 22:20:05,794510,NaT,NaN,,1
3,766657846,1,2026-06-05 00:45:31,2026-06-04 23:41:12,194393,4.650232,None,None,None,None,...,SGC,,scanloc,,2026-06-04 23:42:55,610956,NaT,NaN,,1
4,766659506,1,2026-06-05 00:45:37,2026-06-04 23:41:12,991969,0.289823,None,None,None,None,...,SGC,,william@proc2,,2026-06-04 23:47:54,881863,2026-06-04 23:48:54,356007.0,,1
5,766658211,1,2026-06-05 00:45:34,2026-06-04 23:41:13,386061,3.782928,None,None,None,None,...,SGC,,scanloc,,2026-06-04 23:43:03,585062,NaT,NaN,,1
6,766660716,1,2026-06-05 00:45:41,2026-06-04 23:41:14,131876,0.435875,None,None,None,None,...,SGC,,scanloc,,2026-06-04 23:49:05,516390,NaT,NaN,,1
7,766659341,1,2026-06-05 00:45:37,2026-06-04 23:41:15,441817,2.759321,None,None,None,None,...,SGC,,scanlocbay,,2026-06-04 23:44:45,724249,NaT,NaN,,1
8,766658022,1,2026-06-05 00:45:33,2026-06-04 23:41:15,489174,2.408911,None,None,None,None,...,SGC,,scanloc,,2026-06-04 23:42:57,137011,NaT,NaN,,1
9,766658600,1,2026-06-05 00:45:35,2026-06-04 23:41:16,418940,2.674570,None,None,None,None,...,SGC,,scanlocbay,,2026-06-04 23:43:13,242225,NaT,NaN,,1


To check, as another example, the info from all the origins with ID SGC2026kxlqas can be retrieved by running the following query:

In [38]:
query2 = "Select Origin.time_value, POEv.publicID, Origin.depth_value, Magnitude.magnitude_value, Origin.quality_standardError, Origin.depth_uncertainty, Origin.latitude_uncertainty, Origin.longitude_uncertainty, Origin.quality_associatedPhaseCount, Origin.creationInfo_author, Event.type, Origin.creationInfo_agencyID, EventDescription.text, Origin.latitude_value, Origin.longitude_value, Magnitude.type, Origin.methodID, Origin.earthModelID from Event AS EvMF left join PublicObject AS POEv ON EvMF._oid = POEv._oid left join PublicObject as POOri ON EvMF.preferredOriginID=POOri.publicID left join Origin ON POOri._oid=Origin._oid left join PublicObject as POMag on EvMF.preferredMagnitudeID=POMag.publicID left join Magnitude ON Magnitude._oid = POMag._oid left join Event ON Event._oid= POEv._oid left join EventDescription ON EvMF._oid = EventDescription._parent_oid where Origin.time_value between"
event_df2 = connect_to_db(query2, start_time=dt.datetime(2024, 6, 4, 14, 0, 0), end_time=dt.datetime.now(dt.UTC))
event_df2

2024-06-04 14:00:00 2026-06-05 01:04:34


,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,creationInfo_author,type,creationInfo_agencyID,text,latitude_value,longitude_value,type,methodID,earthModelID
0,2024-06-04 14:01:45,SGC2024kzplve,112.730000,2.243998,0.450000,1.500000,0.848528,0.848528,51.0,jsorianop@proc2,earthquake,SGC,"Betulia - Santander, Colombia",7.055500,-73.478333,MLr_vmm,Hypo71,VMM
1,2024-06-04 16:33:52,SGC2024kzumvy,29.196429,2.414488,1.261676,NaN,4.646398,6.031268,34.0,jbastoa@proc2,earthquake,SGC,"SipÃ­ - ChocÃ³, Colombia",4.465470,-76.436238,MLr_1,NonLinLoc,Poveda_et_al_2018
2,2024-06-04 17:31:46,SGC2024kzwktb,0.000000,1.809123,0.750000,4.900000,1.909188,1.909188,13.0,jbastoa@proc2,explosion,SGC,"Becerrill - Cesar, Colombia",9.695167,-73.482667,MLr_4,Hypo71,modelCesar2
3,2024-06-04 14:25:19,SGC2024kzqgcg,140.195312,2.238412,1.187079,7.602621,3.780325,6.233362,43.0,jbastoa@proc2,earthquake,SGC,"Los Santos - Santander, Colombia",6.807542,-73.151033,MLr_3,NonLinLoc,Poveda_et_al_2018
4,2024-06-04 19:03:56,SGC2024kzzmfe,138.671875,2.092753,0.753254,9.375545,4.175715,7.694378,20.0,jbastoa@proc2,earthquake,SGC,"Los Santos - Santander, Colombia",6.762034,-73.141382,MLr_3,NonLinLoc,Poveda_et_al_2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139397,2026-06-04 09:47:58,SGC2026kxligi,146.800000,2.348788,0.270000,1.200000,0.919239,0.919239,30.0,gerard@proc3,earthquake,SGC,"Los Santos - Santander, Colombia",6.826500,-73.176500,MLr_3,Hypo71,RSNC
139398,2026-06-04 09:57:03,SGC2026kxlqas,145.090000,2.779758,0.440000,1.800000,0.989949,0.989949,37.0,gerard@proc3,earthquake,SGC,"Los Santos - Santander, Colombia",6.830167,-73.146667,MLr_3,Hypo71,RSNC
139399,2026-06-04 12:44:29,SGC2026kxreih,141.355469,2.330239,1.007683,6.019281,3.219354,4.474158,56.0,amarin@proc2,earthquake,SGC,"Los Santos - Santander, Colombia",6.765991,-73.088852,MLr_3,NonLinLoc,Poveda_et_al_2018
139400,2026-06-04 22:18:08,SGC2026kykeqx,285.249939,3.460490,1.758920,14.128080,14.747627,13.960316,5.0,scanloc,None,SGC,"Mani - Casanare, Colombia",4.447649,-72.110695,M,LOCSAT,iasp91


In [39]:
query2 = "SELECT o.time_value, po_e.publicID AS eventID, o.depth_value, m.magnitude_value, o.quality_standardError, o.depth_uncertainty, o.latitude_uncertainty, o.longitude_uncertainty, o.quality_associatedPhaseCount, o.creationInfo_author, e.type AS event_type, o.creationInfo_agencyID, ed.text AS region, o.latitude_value, o.longitude_value, m.type AS mag_type, o.methodID, o.earthModelID FROM Event e JOIN PublicObject po_e ON e._oid = po_e._oid LEFT JOIN PublicObject po_o ON e.preferredOriginID = po_o.publicID LEFT JOIN Origin o ON po_o._oid = o._oid LEFT JOIN PublicObject po_m ON e.preferredMagnitudeID = po_m.publicID LEFT JOIN Magnitude m ON po_m._oid = m._oid LEFT JOIN EventDescription ed ON e._oid = ed._parent_oid WHERE o.time_value BETWEEN"
event_df2 = connect_to_db(query2, start_time=dt.datetime(2024, 6, 4, 14, 0, 0), end_time=dt.datetime.now(dt.UTC))
event_df2

2024-06-04 14:00:00 2026-06-05 01:05:09


,time_value,eventID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,creationInfo_author,event_type,creationInfo_agencyID,region,latitude_value,longitude_value,mag_type,methodID,earthModelID
0,2024-06-04 14:01:45,SGC2024kzplve,112.730000,2.243998,0.450000,1.500000,0.848528,0.848528,51.0,jsorianop@proc2,earthquake,SGC,"Betulia - Santander, Colombia",7.055500,-73.478333,MLr_vmm,Hypo71,VMM
1,2024-06-04 16:33:52,SGC2024kzumvy,29.196429,2.414488,1.261676,NaN,4.646398,6.031268,34.0,jbastoa@proc2,earthquake,SGC,"SipÃ­ - ChocÃ³, Colombia",4.465470,-76.436238,MLr_1,NonLinLoc,Poveda_et_al_2018
2,2024-06-04 17:31:46,SGC2024kzwktb,0.000000,1.809123,0.750000,4.900000,1.909188,1.909188,13.0,jbastoa@proc2,explosion,SGC,"Becerrill - Cesar, Colombia",9.695167,-73.482667,MLr_4,Hypo71,modelCesar2
3,2024-06-04 14:25:19,SGC2024kzqgcg,140.195312,2.238412,1.187079,7.602621,3.780325,6.233362,43.0,jbastoa@proc2,earthquake,SGC,"Los Santos - Santander, Colombia",6.807542,-73.151033,MLr_3,NonLinLoc,Poveda_et_al_2018
4,2024-06-04 19:03:56,SGC2024kzzmfe,138.671875,2.092753,0.753254,9.375545,4.175715,7.694378,20.0,jbastoa@proc2,earthquake,SGC,"Los Santos - Santander, Colombia",6.762034,-73.141382,MLr_3,NonLinLoc,Poveda_et_al_2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139397,2026-06-04 09:47:58,SGC2026kxligi,146.800000,2.348788,0.270000,1.200000,0.919239,0.919239,30.0,gerard@proc3,earthquake,SGC,"Los Santos - Santander, Colombia",6.826500,-73.176500,MLr_3,Hypo71,RSNC
139398,2026-06-04 09:57:03,SGC2026kxlqas,145.090000,2.779758,0.440000,1.800000,0.989949,0.989949,37.0,gerard@proc3,earthquake,SGC,"Los Santos - Santander, Colombia",6.830167,-73.146667,MLr_3,Hypo71,RSNC
139399,2026-06-04 12:44:29,SGC2026kxreih,141.355469,2.330239,1.007683,6.019281,3.219354,4.474158,56.0,amarin@proc2,earthquake,SGC,"Los Santos - Santander, Colombia",6.765991,-73.088852,MLr_3,NonLinLoc,Poveda_et_al_2018
139400,2026-06-04 22:18:08,SGC2026kykeqx,285.249939,3.460490,1.758920,14.128080,14.747627,13.960316,5.0,scanloc,None,SGC,"Mani - Casanare, Colombia",4.447649,-72.110695,M,LOCSAT,iasp91


In [45]:
# Read revision.sql file
with open('./queries/revision.sql', 'r') as file:
    revision_query = file.read()

print(revision_query)

-- SQL query to extract data for revision of the event catalog.
SELECT
    Origin.time_value,  -- Event origin time
    POEv.publicID,  -- Event public ID
    Origin.depth_value,  -- Event depth
    Magnitude.magnitude_value,  -- Event magnitude
    Origin.quality_standardError,  -- Standard error of the origin time
    Origin.depth_uncertainty, -- Uncertainty of the event depth
    Origin.latitude_uncertainty,  -- Uncertainty of the event latitude
    Origin.longitude_uncertainty,  -- Uncertainty of the event longitude
    Origin.quality_associatedPhaseCount,  -- Number of associated phases
    Origin.creationInfo_author,  -- Author of the origin creation info
    Event.type,  -- Event type (e.g., earthquake, explosion)
    Origin.creationInfo_agencyID,  -- Agency ID of the origin creation info
    EventDescription.text,  -- Event description text
    Origin.latitude_value,  -- Event latitude
    Origin.longitude_value,  -- Event longitude
    Magnitude.type,  -- Magnitude type (e.g.,